# H&M Transaction Data: Product Recommendations 01
## Pre-process data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from datetime import datetime

import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

import os
import sys
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# data processing classes
from src.customer_features import CustomerFeatureEngineer
from src.product_features import ProductFeatureEngineer
from src.recommendation_training import RecommendationTrainingBuilder

# Define paths
base_path = Path("../data/")
raw_path = processed_path = base_path / 'raw'
processed_path = base_path / 'processed'

## Load Cleaned Data

In [2]:
customers = pd.read_csv(processed_path / 'customer_hm_cleaned.csv')
transactions = pd.read_csv(processed_path / 'transactions_hm_cleaned.csv')
articles = pd.read_csv(processed_path / 'articles_hm_cleaned.csv')

print("Cleaned Rows of Data:")
print(f"Articles: {len(articles):,}")
print(f"Customers: {len(customers):,}")
print(f"Transactions: {len(transactions):,}")

Cleaned Rows of Data:
Articles: 105,542
Customers: 1,048,575
Transactions: 1,040,101


## Train, Validation Data Generation

Data builded uses a max total number of transaction
It preferentially allocates those transaction to the prediction period and uses a 1:1 ratio of prediction period transaction and pre-prediction transactions

In [3]:
def build_product_recommendation_data(as_of_date, prediction_start_date, prediction_end_date, random_state=67):
    print(f"Data as of date: {as_of_date}")
    print(f"Prediction period: {prediction_start_date} to {prediction_end_date}")

    MAX_TRANSACTIONS = 200000
    max_prediction_transactions = int(MAX_TRANSACTIONS / 2)

    # Find all transactions in the prediction period
    transactions_mask = (transactions['t_dat'] <= prediction_end_date)
    transactions_sample = transactions[transactions_mask]
    print(f"Transaction rows available: {len(transactions_sample):,}")

    # Transactions in prediction period
    prediction_period_mask = (transactions_sample['t_dat'] >= prediction_start_date) & (transactions_sample['t_dat'] <= prediction_end_date)
    
    transactions_prediction_period = transactions_sample[prediction_period_mask]
    prediction_transactions_count = len(transactions_prediction_period)
    transactions_prediction_period =  transactions_prediction_period.sample(n=min(max_prediction_transactions,prediction_transactions_count), random_state=random_state)
    prediction_transactions_count = len(transactions_prediction_period)
    print(f"Transaction rows in prediction period: {len(transactions_prediction_period):,}")

    # Remaining slots allocated to pre-prediction transactions
    transactions_before_prediction_period_count = min(int(MAX_TRANSACTIONS - prediction_transactions_count), prediction_transactions_count)

    # Transactions before prediction period
    as_of_mask = (transactions['t_dat'] < prediction_start_date)
    transactions_before_prediction = transactions[as_of_mask]
    print(f"Transaction rows before prediction: {len(transactions_before_prediction):,}")
    transactions_before_prediction = transactions_before_prediction.sample(n=transactions_before_prediction_period_count, random_state=random_state)
    print(f"Transaction rows in as_of_date after sampling: {len(transactions_before_prediction):,}")

    transactions_df = pd.concat([transactions_prediction_period, transactions_before_prediction])
    print(f"Transaction rows: {len(transactions_df):,}")
    
    # Pull all customers and articles that are in the sampled transactions
    sample_customer_ids = transactions_df['customer_id'].unique()
    sample_article_ids = transactions_df['article_id'].unique()
    
    customers_sample = customers[customers['customer_id'].isin(sample_customer_ids)]
    articles_sample = articles[articles['article_id'].isin(sample_article_ids)]
    
    customers_df = customers_sample
    articles_df = articles_sample

    
    cfe = CustomerFeatureEngineer(customers_df=customers_df, transactions_df=transactions_df)
    customer_features = cfe.calculate_all_features(articles_df=articles_df, as_of_date=as_of_date)
    print(f"Customer rows: {len(customer_features):,}")

    pfe = ProductFeatureEngineer(articles_df=articles_df, transactions_df=transactions_df)
    product_features = pfe.calculate_all_features(as_of_date=as_of_date)
    print(f"Product rows: {len(product_features):,}")

    builder = RecommendationTrainingBuilder(transactions_df=transactions_df,
                                                customer_features_df=customer_features,
                                                product_features_df=product_features)
    data = builder.build_dataset(prediction_start=prediction_start_date,
                                            prediction_end=prediction_end_date,
                                            negative_ratio=5,
                                            random_state=random_state)

    print(f"Total rows: {len(data):,}")
    print(f"- Positives: {(data['purchased'] == 1).sum():,}")
    print(f"- Negatives: {(data['purchased'] == 0).sum():,}")
    return data

In [4]:
print("=========TRAINING DATA=========")
train_as_of_date = "2019-09-30"
train_prediction_start_date = "2019-10-01"
train_prediction_end_date = "2019-10-30"
train_data =  build_product_recommendation_data(train_as_of_date, train_prediction_start_date, train_prediction_end_date)

=========TRAINING DATA=========
Data as of date: 2019-09-30
Prediction period: 2019-10-01 to 2019-10-30
Transaction rows available: 891,243
Transaction rows in prediction period: 71,116
Transaction rows before prediction: 820,127
Transaction rows in as_of_date after sampling: 71,116
Transaction rows: 142,232
Customer rows: 63,806
Product rows: 21,003
Total rows: 44,844
- Positives: 7,474
- Negatives: 37,370


In [5]:
print("\n=========VALIDATION DATA=========")
val_as_of_date = "2019-10-31"
val_prediction_start = "2019-11-01"
val_prediction_end = "2019-11-30"
val_data =  build_product_recommendation_data(val_as_of_date, val_prediction_start, val_prediction_end)


=========VALIDATION DATA=========
Data as of date: 2019-10-31
Prediction period: 2019-11-01 to 2019-11-30
Transaction rows available: 969,160
Transaction rows in prediction period: 75,774
Transaction rows before prediction: 893,386
Transaction rows in as_of_date after sampling: 75,774
Transaction rows: 151,548
Customer rows: 67,796
Product rows: 22,151
Total rows: 51,498
- Positives: 8,583
- Negatives: 42,915


In [6]:
print("\n=========TEST DATA=========")
test_as_of_date = "2019-11-30"
test_prediction_start = "2019-12-01"
test_prediction_end = "2019-12-31"
test_data =  build_product_recommendation_data(test_as_of_date, test_prediction_start, test_prediction_end)


=========TEST DATA=========
Data as of date: 2019-11-30
Prediction period: 2019-12-01 to 2019-12-31
Transaction rows available: 1,040,101
Transaction rows in prediction period: 70,941
Transaction rows before prediction: 969,160
Transaction rows in as_of_date after sampling: 70,941
Transaction rows: 141,882
Customer rows: 64,113
Product rows: 22,161
Total rows: 42,006
- Positives: 7,001
- Negatives: 35,005


In [7]:
feature_cols = [col for col in train_data.columns if col not in ['customer_id', 'article_id', 'purchased']]

print(f"Feature Columns:")
for col in feature_cols:
    print(f"- {col}")


Feature Columns:
- sales_last_7_days
- sales_last_30_days
- days_since_first_sale
- days_since_last_sale
- avg_price
- min_price
- max_price
- product_price_std
- customer_price_std
- num_purchases
- total_spent
- days_since_last_purchase
- avg_transaction_value
- avg_days_between_purchases
- primary_department
- primary_garment_group
- category_diversity


### Categorical features
One hot encode the garment groups. Drop primary department, too many possibilities

In [8]:
print(f"Unique departments: {train_data['primary_department'].nunique()}")
print(f"Unique garment groups: {train_data['primary_garment_group'].nunique()}")

Unique departments: 183
Unique garment groups: 21


In [9]:
train_data = train_data.drop('primary_department', axis=1)
val_data = val_data.drop('primary_department', axis=1)
test_data = test_data.drop('primary_department', axis=1)

train_data = pd.get_dummies(train_data, columns=['primary_garment_group'], prefix='garment')
val_data = pd.get_dummies(val_data, columns=['primary_garment_group'], prefix='garment')
test_data = pd.get_dummies(test_data, columns=['primary_garment_group'], prefix='garment')


all_columns = set(train_data.columns).union(val_data.columns).union(test_data.columns)
print(all_columns)

train_data = train_data.reindex(columns=all_columns, fill_value=0)
val_data = val_data.reindex(columns=all_columns, fill_value=0)
test_data = test_data.reindex(columns=all_columns, fill_value=0)

{'avg_days_between_purchases', 'min_price', 'category_diversity', 'purchased', 'garment_Dressed', 'max_price', 'garment_Jersey Basic', 'garment_Accessories', 'sales_last_30_days', 'customer_id', 'days_since_first_sale', 'garment_Swimwear', 'num_purchases', 'garment_Unknown', 'garment_Skirts', 'garment_Jersey Fancy', 'garment_Special Offers', 'days_since_last_sale', 'garment_Knitwear', 'garment_Trousers Denim', 'garment_Dresses Ladies', 'garment_Outdoor', 'garment_Woven/Jersey/Knitted mix Baby', 'garment_Socks and Tights', 'garment_Blouses', 'total_spent', 'garment_Dresses/Skirts girls', 'avg_price', 'avg_transaction_value', 'garment_Shorts', 'garment_Trousers', 'garment_Shoes', 'article_id', 'garment_Under-, Nightwear', 'days_since_last_purchase', 'sales_last_7_days', 'product_price_std', 'customer_price_std', 'garment_Shirts'}


### X, y split
Outcome variable is `purchased` and 0 or 1 feature

In [10]:
X_train = train_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_train = train_data['purchased']
print(f"{X_train.shape=}")
print(f"{y_train.shape=}")
X_val = val_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_val = val_data['purchased']
print(f"{X_val.shape=}")
print(f"{y_val.shape=}")
X_test = test_data.drop(['customer_id', 'article_id', 'purchased'], axis=1)
y_test = test_data['purchased']
print(f"{X_test.shape=}")
print(f"{y_test.shape=}")

X_train.shape=(44844, 36)
y_train.shape=(44844,)
X_val.shape=(51498, 36)
y_val.shape=(51498,)
X_test.shape=(42006, 36)
y_test.shape=(42006,)


## Save Processed Data
### as pickle files

In [11]:
# Save data as pickle to avoid reprocessing
processed_data_path = processed_path / 'product_recommendation'
processed_data_path.mkdir(parents=True, exist_ok=True)

with open(processed_data_path / 'X_train_base.pkl', 'wb') as f:
    pickle.dump(X_train, f)
with open(processed_data_path / 'y_train_base.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open(processed_data_path / 'X_val_base.pkl', 'wb') as f:
    pickle.dump(X_val, f)
with open(processed_data_path / 'y_val_base.pkl', 'wb') as f:
    pickle.dump(y_val, f)
with open(processed_data_path / 'X_test_base.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open(processed_data_path / 'y_test_base.pkl', 'wb') as f:
    pickle.dump(y_test, f)